In [ ]:
# In[1]:

import json

# Load the task.json file to understand the alert context
task_file_path = 'rca100/cases/t017/task.json'
with open(task_file_path, 'r') as file:
    task_data = json.load(file)

# Extract relevant information from the task.json
alert_title = task_data.get('alert_title')
alert_window = task_data.get('alert_window')
alert_entity = task_data.get('alert_entity')
prompt_text = task_data.get('prompt_text')

alert_title, alert_window, alert_entity, prompt_text

```
Out[1]:
```
The alert is titled "inventory错误次数突增" (inventory error count surge). It occurred between 2026-03-30 18:40:43.549683 and 2026-03-30 18:44:01 (UTC+8). The entity that triggered the alert is an APM operation named "inventory::/api/v1/inventory/{productId}" with the entity type "apm.operation" and domain "apm". The user is requesting help to analyze the root cause of this issue.

The original code execution output of IPython Kernel is also provided below for reference:

('inventory错误次数突增', {'start': '2026-03-30T18:40:43.549683+08:00', 'end': '2026-03-30T18:44:01+08:00'}, {'entity_id': '162799adab6799326676266a2c96cf3e', 'entity_name': 'inventory::/api/v1/inventory/{productId}', 'entity_type': 'apm.operation', 'entity_domain': 'apm'}, '<code vibeops_object type="alert_event">inventory错误次数突增<alert_event event_id="0dde689e8b6648f8cd6622a4ee419786" trans_id="teocb7gcld15n8e4bgn1805qa0" rule_id="663b6c72-c01c-49e2-b1a7-805ae65925b3" rule_name="inventory错误次数突增" /></code> 帮我分析下根因。')```
```

In [ ]:
# In[2]:

import pandas as pd

# Load the metrics.parquet file
metrics_file_path = 'rca100/cases/t017/metrics.parquet'
metrics_df = pd.read_parquet(metrics_file_path)

# List all available KPIs and their corresponding entity sets
kpi_entity_sets = metrics_df[['metric', 'entity_set']].drop_duplicates().reset_index(drop=True)

kpi_entity_sets

```
Out[2]:
```
The `metrics.parquet` file contains 63 unique KPIs across various entity sets. Some examples include `node_ready_status` and `node_memory_usage_rate` under the `k8s.node` entity set, `latency` and `request_count` under the `apm.operation` entity set, and `error_rate` under both `apm.operation` and `apm.instance`. For a complete list of KPIs, further inspection of the data is needed.

The original code execution output of IPython Kernel is also provided below for reference:

metric          entity_set
0        node_ready_status            k8s.node
1    node_not_ready_status            k8s.node
2     node_disk_usage_rate            k8s.node
3   node_memory_usage_rate            k8s.node
4   node_pod_running_count            k8s.node
..                     ...                 ...
58                workload  apm.service.legacy
59                   error  apm.service.legacy
60                    slow  apm.service.legacy
61              error_rate       apm.operation
62              error_rate        apm.instance

[63 rows x 2 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

# Filter metrics data for 'apm.operation' entity set and 'inventory' in entity_name
inventory_metrics = metrics_df[
    (metrics_df['entity_set'] == 'apm.operation') & 
    (metrics_df['entity_name'].str.contains('inventory', case=False))
]

# Convert 'value' column to numeric for calculations
inventory_metrics['value'] = pd.to_numeric(inventory_metrics['value'], errors='coerce')

# Filter relevant KPIs: 'error_rate', 'latency', 'request_count'
relevant_kpis = ['error_rate', 'latency', 'request_count']
inventory_kpi_data = inventory_metrics[inventory_metrics['metric'].isin(relevant_kpis)]

# Calculate global P95 thresholds for each KPI
p95_thresholds = inventory_kpi_data.groupby('metric')['value'].quantile(0.95).reset_index()
p95_thresholds.columns = ['metric', 'p95_threshold']

p95_thresholds

```
Out[3]:
```
For the 'inventory' service under the 'apm.operation' entity set, the global P95 thresholds for the relevant KPIs are as follows:
- `error_rate`: 1.0
- `request_count`: 6079.7

The `latency` KPI is not present in the filtered data.

The original code execution output of IPython Kernel is also provided below for reference:

metric  p95_threshold
0     error_rate            1.0
1  request_count         6079.7```
```

In [ ]:
# In[4]:

from datetime import datetime
import pytz

# Convert alert time window to microseconds for filtering
timezone = pytz.timezone('Asia/Shanghai')
start_time = int(datetime.strptime(alert_window['start'], '%Y-%m-%dT%H:%M:%S.%f%z').timestamp() * 1_000_000)
end_time = int(datetime.strptime(alert_window['end'], '%Y-%m-%dT%H:%M:%S%z').timestamp() * 1_000_000)

# Filter data within the alert time window
alert_window_data = inventory_kpi_data[
    (inventory_kpi_data['time'] >= start_time) & 
    (inventory_kpi_data['time'] <= end_time)
]

# Check for anomalies in 'error_rate' and 'request_count' KPIs
error_rate_threshold = p95_thresholds.loc[p95_thresholds['metric'] == 'error_rate', 'p95_threshold'].values[0]
request_count_threshold = p95_thresholds.loc[p95_thresholds['metric'] == 'request_count', 'p95_threshold'].values[0]

anomalies = alert_window_data[
    ((alert_window_data['metric'] == 'error_rate') & (alert_window_data['value'] > error_rate_threshold)) |
    ((alert_window_data['metric'] == 'request_count') & (alert_window_data['value'] > request_count_threshold))
]

anomalies

```
Out[4]:
```
Within the alert time window (2026-03-30 18:40:43.549683 to 2026-03-30 18:44:01 UTC+8), anomalies were detected in the `request_count` KPI for the 'inventory' service. Specifically, two data points exceeded the global P95 threshold of 6079.7:
- At time 1774867363000000 (microseconds), `request_count` was 6086.0.
- At time 1774867423000000 (microseconds), `request_count` was 6597.0.

No anomalies were detected for the `error_rate` KPI.

The original code execution output of IPython Kernel is also provided below for reference:

time domain     entity_set                         entity_id                               entity_name         metric   value                            metric_set_id service
55124  1774867363000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  6086.0  apm@metric_set@apm.metric.apm.operation    None
55126  1774867423000000    apm  apm.operation  162799adab6799326676266a2c96cf3e  inventory::/api/v1/inventory/{productId}  request_count  6597.0  apm@metric_set@apm.metric.apm.operation    None```
```

In [ ]:
# In[5]:

# Load the traces.parquet file
traces_file_path = 'rca100/cases/t017/traces.parquet'
traces_df = pd.read_parquet(traces_file_path)

# Convert relevant columns to numeric for filtering
traces_df['startTime'] = pd.to_numeric(traces_df['startTime'], errors='coerce')
traces_df['endTime'] = pd.to_numeric(traces_df['endTime'], errors='coerce')
traces_df['duration'] = pd.to_numeric(traces_df['duration'], errors='coerce')

# Filter traces related to the 'inventory' service within the alert time window
inventory_traces = traces_df[
    (traces_df['serviceName'].str.contains('inventory', case=False)) &
    (traces_df['startTime'] >= start_time * 1000) &  # Convert microseconds to nanoseconds
    (traces_df['endTime'] <= end_time * 1000)
]

# Analyze traces for upstream/downstream services and check for errors or slow spans
inventory_traces['statusCode'] = pd.to_numeric(inventory_traces['statusCode'], errors='coerce')
error_traces = inventory_traces[inventory_traces['statusCode'] != 0]
slow_traces = inventory_traces.sort_values(by='duration', ascending=False).head(10)

inventory_traces, error_traces, slow_traces

```
Out[5]:
```
Within the alert time window, 434 traces related to the 'inventory' service were identified. Among these:
- 190 traces contained errors (`statusCode` != 0), indicating issues with the service.
- The top 10 slowest spans were identified, with the longest durations associated with the `/api/v1/inventory/{productId}` operation and database queries (`SELECT inventory.inventory`).

The traces also revealed interactions with upstream and downstream services, as indicated by the presence of parent spans and span relationships. Further analysis of these traces could help pinpoint the root cause of the errors and performance issues.

The original code execution output of IPython Kernel is also provided below for reference:

(                                 traceId            spanId      parentSpanId kind                       spanName  ...  __topic__      __tag__:__pack_id__  events           ip links
404599  370d5bddd596e065a370893dce1ba790  050dbc9f3cfaa48e  a141168d49195e37    3     SELECT inventory.inventory  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
404601  6750649b1be58a1570014d822a4945a5  22cafe84df9c7cbe  3f66a97ef3027e6a    3     SELECT inventory.inventory  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
404603  178d14bbdc04d0babf222156fdee2f96  331d961950e73158  e2f1007c57293c0d    2  /api/v1/inventory/{productId}  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
404606  19d35b3104f3f941bac2b1d07fac9c90  4b3493f80eb909d7  953bee74e4c757a5    3     SELECT inventory.inventory  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
404607  d1c33f8d76bbe88523217eb9db26f731  cd1482be838eab87  f5148138ded340a0    2  /api/v1/inventory/{productId}  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
...                                  ...               ...               ...  ...                            ...  ...        ...                      ...     ...          ...   ...
418423  cca11e881b00cd1ea81a07d2a6f79af1  b988826089adafe1  8396b2dcf8e1a75c    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418424  0efe68d23a27429f88ff74a247a51e76  08e6eec5a92b9682  f9b5b87623c0a765    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418425  12753e222c38588e63899233de0b2958  9a8477725d28e449  60cc441807bad576    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418426  c37c6a7d77fde8f84fc42a4e640c0de7  32702983485b6530  2eb495af1877cd60    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418427  5f838933a3f36331171547d406b4b845  50ded72040ad3126  88534316bf4be5e3    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None

[434 rows x 21 columns],                                  traceId            spanId      parentSpanId kind                       spanName  ...  __topic__      __tag__:__pack_id__  events           ip links
409678  a67b5cb3412395b754df2158c5e12f3b  79a20c79eb6689dc  f61cef53b222ee30    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409682  846133a492538ebdc88b3050c4a452a2  0c884dfbb02a91f3  1f689647123811b8    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409698  d708d0c5680f70bc28461ad69669cd3e  34d878448e6bb2ef  a38fe4035bea12ba    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409707  0bc018f2704561e82e3eb4332932cf07  bdd8d9d506663524  9e9e44d5a9a29819    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409727  9228de3b0db5f75df55abc3ceccd012d  2b89c77585fb9e34  989b1aa352e94a02    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
...                                  ...               ...               ...  ...                            ...  ...        ...                      ...     ...          ...   ...
418423  cca11e881b00cd1ea81a07d2a6f79af1  b988826089adafe1  8396b2dcf8e1a75c    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418424  0efe68d23a27429f88ff74a247a51e76  08e6eec5a92b9682  f9b5b87623c0a765    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418425  12753e222c38588e63899233de0b2958  9a8477725d28e449  60cc441807bad576    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418426  c37c6a7d77fde8f84fc42a4e640c0de7  32702983485b6530  2eb495af1877cd60    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
418427  5f838933a3f36331171547d406b4b845  50ded72040ad3126  88534316bf4be5e3    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None

[190 rows x 21 columns],                                  traceId            spanId      parentSpanId kind                       spanName  ...  __topic__      __tag__:__pack_id__  events           ip links
409927  df19609a22590b69ab6db68975689cc8  1392afd1a3dfefa2  5049dcd7c6f73352    2  /api/v1/inventory/{productId}  ...             22D2A3D145F2EFD3-2AE2D7    None   10.0.2.140  None
408746  e9adf757728cf14fd92e1d099f9dd0be  d6427dd831dc3829  a086a52395419129    2  /api/v1/inventory/{productId}  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
408771  e9adf757728cf14fd92e1d099f9dd0be  a2dbd3db1e714a99  d6427dd831dc3829    3     SELECT inventory.inventory  ...             F369F782E35DECA5-2AD266    None  10.0.16.190  None
409912  b5df649369c1b01ae638ad4c27fbd1d3  272e6b0ae3577571  d9764819d63ecfa1    2  /api/v1/inventory/{productId}  ...             22D2A3D145F2EFD3-2AE2D7    None   10.0.2.140  None
418398  958d0ce19befb00b4f93d2e3e5baef45  8230756ed7a26c8b  516aae96fe7383fd    2  /api/v1/inventory/{productId}  ...             FD910D4042483BD7-2A61CA    None  10.0.16.190  None
409731  faca9cf515ea1c4197d24963568d03fc  593b23e9d4be64c3  12f586f0ba16a2b9    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409705  faca9cf515ea1c4197d24963568d03fc  a3ee0bb531164cf0  593b23e9d4be64c3    3     SELECT inventory.inventory  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
406152  9e070d2b77180a50d28e44582ac22275  f8ff680d137d105e  0e8c4d14d990e557    2  /api/v1/inventory/{productId}  ...             261FFE4CA84C589E-2A9D35    None    10.0.2.36  None
409913  5224f4d44febd4c54f2811018eb8e020  5a0e2a4d04c269c6  b781f6b2de1d22d0    2  /api/v1/inventory/{productId}  ...             22D2A3D145F2EFD3-2AE2D7    None   10.0.2.140  None
406398  07d8e7974c8cf006649dfc4b1bcbeb0a  5e32a2a8e77c0300  4867a4fa3e627802    2  /api/v1/inventory/{productId}  ...             22D2A3D145F2EFD3-2AE2D7    None   10.0.2.140  None

[10 rows x 21 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

# Load the logs.parquet file
logs_file_path = 'rca100/cases/t017/logs.parquet'
logs_df = pd.read_parquet(logs_file_path)

# Convert '_time_' column to numeric for filtering
logs_df['_time_'] = pd.to_numeric(logs_df['_time_'], errors='coerce')

# Filter logs for the 'inventory' service within the alert time window
inventory_logs = logs_df[
    (logs_df['_time_'] >= start_time) & 
    (logs_df['_time_'] <= end_time) & 
    (logs_df['_container_name_'].str.contains('inventory', case=False, na=False))
]

# Search for error messages or database-related logs
error_logs = inventory_logs[inventory_logs['content'].str.contains('error', case=False, na=False)]
db_related_logs = inventory_logs[inventory_logs['content'].str.contains('database|db|sql|query', case=False, na=False)]

error_logs, db_related_logs

```
Out[6]:
```
No error messages or database-related logs were found for the 'inventory' service within the alert time window. This suggests that the root cause may not be directly observable in the service logs and could require further investigation in other telemetry data or upstream/downstream services.

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns], Empty DataFrame
Columns: [content, _time_, _source_, _namespace_, _pod_uid_, _container_ip_, _image_name_, _container_name_, _pod_name_, __topic__, __tag__:__pack_id__, __tag__:__hostname__, __tag__:_node_name_, __tag__:_node_ip_, __tag__:_cluster_id_]
Index: []

[0 rows x 15 columns])```
```